# Lesson 01 - Εισαγωγή στους Πράκτορες Τεχνητής Νοημοσύνης

Καλώς ήρθατε στο πρώτο μάθημα του μαθήματος **Πράκτορες Τεχνητής Νοημοσύνης για Αρχάριους**!

Ένας **πράκτορας ΤΝ** είναι ένα πρόγραμμα που χρησιμοποιεί ένα μεγάλο γλωσσικό μοντέλο (LLM) ως μηχανή επιχειρηματολογίας και μπορεί να εκτελεί *ενέργειες* στον πραγματικό κόσμο — να καλεί APIs, να ερωτά βάσεις δεδομένων ή να τρέχει κώδικα — για να εκπληρώσει έναν στόχο εκ μέρους ενός χρήστη.

Σε αυτό το σημειωματάριο θα δημιουργήσετε τον πρώτο σας πράκτορα: έναν **Πράκτορα Ταξιδιών** που προτείνει προορισμούς διακοπών. Κατά τη διάρκεια θα μάθετε πώς να:

1. Συνδεθείτε με την Υπηρεσία Πράκτορα Azure AI Foundry χρησιμοποιώντας το **Microsoft Agent Framework**.
2. Δώσετε στον πράκτορα ένα **εργαλείο** — μια απλή συνάρτηση Python που μπορεί να καλέσει.
3. Τρέξετε τον πράκτορα και να εξετάσετε την απάντησή του.
4. Μεταδώστε την απάντηση του πράκτορα διαδοχικά, token-ανά-token.


## Ρύθμιση

Πριν εκτελέσετε αυτό το σημειωματάριο, βεβαιωθείτε ότι έχετε:

1. **Ένα έργο Azure AI Foundry** με αναπτυγμένο μοντέλο συνομιλίας (π.χ. `gpt-4o-mini`).
2. **Συνδεθείτε με το Azure CLI** — εκτελέστε `az login` στο τερματικό σας.
3. **Ορίστε τις απαιτούμενες μεταβλητές περιβάλλοντος:**
   - `AZURE_AI_PROJECT_ENDPOINT` — το σημείο σύνδεσης του έργου Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — το όνομα του αναπτυγμένου μοντέλου σας.

Το κελί παρακάτω εγκαθιστά τα πακέτα Python που χρειάζεστε.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Δημιουργία του Πρώτου σας Πράκτορα

Ένας πράκτορας χρειάζεται δύο πράγματα:

- **Οδηγίες** που του λένε *ποιος είναι* και *πώς να συμπεριφέρεται* (ένα σύστημα εντολών).
- **Εργαλεία** — συναρτήσεις Python διακοσμημένες με `@tool` που ο πράκτορας μπορεί να καλέσει για να ανακτήσει πληροφορίες ή να εκτελέσει ενέργειες.

Παρακάτω ορίζουμε ένα απλό εργαλείο που επιστρέφει μια λίστα με δημοφιλείς προορισμούς διακοπών. Ο πράκτορας θα χρησιμοποιήσει αυτό το εργαλείο όταν ένας χρήστης ζητήσει ταξιδιωτικές προτάσεις.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Για μια πιο διαδραστική εμπειρία, μπορείτε να **ροής** την απάντηση του πράκτορα. Αντί να περιμένετε για την πλήρη απάντηση, ο πράκτορας παρέχει τμήματα κειμένου καθώς αυτά παράγονται. Αυτό είναι ιδιαίτερα χρήσιμο σε περιβάλλοντα συνομιλίας όπου θέλετε να εμφανίζετε την έξοδο σε πραγματικό χρόνο.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Σύνοψη

Σε αυτό το μάθημα μάθατε πώς να:

- **Δημιουργήσετε έναν πάροχο** που συνδέεται με την υπηρεσία Azure AI Foundry Agent μέσω του `AzureAIProjectAgentProvider`.
- **Ορίσετε ένα εργαλείο** χρησιμοποιώντας το διακοσμητή `@tool` ώστε ο πράκτορας να μπορεί να καλεί τις συναρτήσεις Python σας.
- **Τρέξετε τον πράκτορα** με ένα μήνυμα χρήστη και να εκτυπώσετε την απάντησή του.
- **Ρεύμα απαντήσεων** για έξοδο σε πραγματικό χρόνο.

Στο επόμενο μάθημα θα εξερευνήσουμε πιο βαθιά τα agentic frameworks και θα μάθουμε πώς να δίνουμε στους πράκτορες πιο ισχυρά εργαλεία και δυνατότητες λογικής πολλαπλών βημάτων.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση Ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται η επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
